# Segmenter BTXRD: continue epochs 51?100

Load the final epoch-50 weights and use a newly initialized AdamW optimizer with a lower learning rate. The previous best validation checkpoint is preserved if no later epoch improves it.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

assert Path('/kaggle/working').is_dir(), 'This notebook must run on Kaggle.'
os.environ['HF_HOME'] = '/kaggle/temp/huggingface'
os.environ['PYTHONUNBUFFERED'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.31', 'einops==0.8.1'], check=True)
import timm
import torch
from huggingface_hub import hf_hub_download

assert torch.cuda.is_available(), 'CUDA GPU unavailable: check the Kaggle accelerator and GPU quota.'
print('Torch:', torch.__version__, 'timm:', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0), flush=True)


In [ ]:
import hashlib

DATASET_REVISION = 'cf8b0569b18e43e7cecab412a06e60aeaf258b32'
EXPECTED_SHA256 = '59ce94244fdf997f0c68e6a378690a4359e6c411f7fda7ac2f3994489637884b'
archive = Path(hf_hub_download(repo_id='torusama/xla-seminar-uia', repo_type='dataset',
                               filename='btxrd_final.zip', revision=DATASET_REVISION))
digest = hashlib.sha256(archive.read_bytes()).hexdigest()
if digest != EXPECTED_SHA256:
    raise RuntimeError(f'Dataset checksum mismatch: {digest}')
print('BTXRD downloaded; SHA-256:', digest, flush=True)


In [ ]:
# Official Segmenter decoder is bundled with its MIT license.
PROJECT = Path('/kaggle/temp/xla-segmenter-continue')
SOURCES = {'src/xla/__init__.py': '"""Research scaffold for vision classification and segmentation."""\n\n__version__ = "0.1.0"\n', 'src/xla/data/__init__.py': '"""Data interfaces for future phases."""\n', 'src/xla/data/hf_segmentation.py': '"""Read the pinned Hugging Face archive and rasterize polygon masks."""\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path\nfrom zipfile import ZipFile\n\nimport torch\nfrom PIL import Image, ImageDraw\nfrom torch.utils.data import Dataset\nfrom torchvision.transforms import functional as TF\n\n\nclass PolygonArchive(Dataset):\n    """Binary union of all annotated polygons; normal samples have empty masks."""\n\n    def __init__(self, archive: Path, split: str, excluded_hashes: set[str] | None = None):\n        self.archive = Path(archive)\n        self.samples = []\n        self.excluded = 0\n        self._zip = None\n        self._zip_pid = None\n        excluded_hashes = excluded_hashes or set()\n        with ZipFile(self.archive) as z:\n            names = set(z.namelist())\n            annotations = sorted(n for n in names if n.startswith(f"{split}/annotations/")\n                                 and n.endswith(".json"))\n            for name in annotations:\n                annotation = json.loads(z.read(name))\n                category = Path(name).parent.name\n                image_name = f"{split}/images/{category}/{annotation[\'image_name\']}"\n                if image_name not in names:\n                    raise ValueError(f"Missing image for {name}: {image_name}")\n                digest = hashlib.sha256(z.read(image_name)).hexdigest()\n                if digest in excluded_hashes:\n                    self.excluded += 1\n                    continue\n                self.samples.append((image_name, name, digest))\n        if not self.samples:\n            raise ValueError(f"No samples in {self.archive} split {split}")\n\n    @property\n    def hashes(self) -> set[str]:\n        return {sample[2] for sample in self.samples}\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getstate__(self):\n        state = self.__dict__.copy()\n        state["_zip"] = None\n        state["_zip_pid"] = None\n        return state\n\n    def __getitem__(self, index):\n        image_name, annotation_name, _ = self.samples[index]\n        if self._zip is None or self._zip_pid != os.getpid():\n            if self._zip is not None:\n                self._zip.close()\n            self._zip = ZipFile(self.archive)\n            self._zip_pid = os.getpid()\n        annotation = json.loads(self._zip.read(annotation_name))\n        with Image.open(io.BytesIO(self._zip.read(image_name))) as source:\n            image = source.convert("RGB")\n        expected = (annotation["image_width"], annotation["image_height"])\n        if image.size != expected or expected != (224, 224):\n            raise ValueError(f"Image and annotation dimensions do not match 224x224: {image_name}")\n        mask = Image.new("L", image.size, 0)\n        draw = ImageDraw.Draw(mask)\n        count = 0\n        for item in annotation["annotations"]:\n            for polygon in item.get("polygons", []):\n                if len(polygon) < 3:\n                    raise ValueError(f"Invalid polygon: {annotation_name}")\n                draw.polygon([tuple(point) for point in polygon], fill=1)\n                count += 1\n        if not count and annotation["class_label"].lower() != "normal":\n            raise ValueError(f"Non-normal sample has no polygons: {annotation_name}")\n        pixels = TF.normalize(TF.to_tensor(image), mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))\n        target = torch.frombuffer(bytearray(mask.tobytes()), dtype=torch.uint8).reshape(1, 224, 224)\n        return pixels, target.float()\n\n\ndef make_splits(archive: Path):\n    """Remove conflicting labels and train duplicates of held-out images."""\n    splits = [PolygonArchive(archive, name) for name in ("train", "val", "test")]\n    categories = {}\n    for split in splits:\n        for image_name, _, digest in split.samples:\n            categories.setdefault(digest, set()).add(Path(image_name).parent.name)\n    conflicts = {digest for digest, labels in categories.items() if len(labels) > 1}\n    for split in splits:\n        original = len(split.samples)\n        split.samples = [sample for sample in split.samples if sample[2] not in conflicts]\n        split.excluded = original - len(split.samples)\n    train, val, test = splits\n    if val.hashes & test.hashes:\n        raise ValueError("Validation and test contain identical images; resolve the split first")\n    held_out = val.hashes | test.hashes\n    original = len(train.samples)\n    train.samples = [sample for sample in train.samples if sample[2] not in held_out]\n    train.excluded += original - len(train.samples)\n    if any(not split.samples for split in splits):\n        raise ValueError("A split is empty after removing conflicting and duplicate images")\n    return train, val, test\n', 'src/xla/engine/__init__.py': '"""Engine interfaces for future phases."""\n', 'src/xla/engine/vit_segmentation.py': '"""Training and evaluation utilities for binary ViT segmentation."""\nimport torch\nfrom torch.nn import functional as F\n\n\ndef segmentation_loss(logits, target, positive_weight=10.0):\n    probabilities = logits.sigmoid()\n    if positive_weight is None:\n        intersection = (probabilities * target).sum(dim=(1, 2, 3))\n        denominator = probabilities.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))\n        return F.binary_cross_entropy_with_logits(logits, target) + (\n            1 - ((2 * intersection + 1) / (denominator + 1)).mean()\n        )\n    intersection = (probabilities * target).sum()\n    denominator = probabilities.sum() + target.sum()\n    dice_loss = 1 - (2 * intersection + 1) / (denominator + 1)\n    weight = logits.new_tensor(positive_weight)\n    return F.binary_cross_entropy_with_logits(logits, target, pos_weight=weight) + dice_loss\n\n\ndef run_epoch(model, loader, device, optimizer=None, positive_weight=10.0):\n    training = optimizer is not None\n    model.train(training)\n    total_loss = 0.0\n    tp = fp = fn = 0\n    count = 0\n    with torch.set_grad_enabled(training):\n        for images, target in loader:\n            images, target = images.to(device), target.to(device)\n            if training:\n                optimizer.zero_grad(set_to_none=True)\n            logits = model(images)\n            loss = segmentation_loss(logits, target, positive_weight)\n            if training:\n                loss.backward()\n                optimizer.step()\n            batch = images.shape[0]\n            total_loss += loss.item() * batch\n            count += batch\n            prediction = logits.detach() >= 0\n            truth = target.bool()\n            tp += (prediction & truth).sum().item()\n            fp += (prediction & ~truth).sum().item()\n            fn += (~prediction & truth).sum().item()\n    return {"loss": total_loss / count, "dice": (2 * tp / (2 * tp + fp + fn)) if tp + fp + fn else 1.0,\n            "iou": (tp / (tp + fp + fn)) if tp + fp + fn else 1.0}\n', 'src/xla/models/__init__.py': '"""Models interfaces for future phases."""\n', 'src/xla/models/vit.py': '"""Thin adapter for the original ImageNet-21k ViT-B/16 PyTorch port."""\n\nMODEL_NAME = "vit_base_patch16_224.orig_in21k"\n\n\ndef build_vit(*, pretrained: bool = True, num_classes: int = 2):\n    """Create timm ViT; pretrained backbone may be downloaded by timm.\n\n    The checkpoint has no classification head, so timm initializes a new head.\n    pretrained=False is intended for offline tests, not a pretrained experiment.\n    """\n    if num_classes < 1:\n        raise ValueError("num_classes must be positive")\n    try:\n        import timm\n    except ImportError as exc:\n        raise RuntimeError(\'Install the ViT extra: python -m pip install -e ".[vit]"\') from exc\n    try:\n        return timm.create_model(MODEL_NAME, pretrained=pretrained, num_classes=num_classes)\n    except Exception as exc:\n        mode = "pretrained weights" if pretrained else "model"\n        raise RuntimeError(f"Could not load {mode} for {MODEL_NAME}: {exc}") from exc\n\n\ndef preprocessing_config(model) -> dict:\n    """Return the model\'s actual timm preprocessing settings."""\n    from timm.data import resolve_model_data_config\n\n    return resolve_model_data_config(model)\n', 'src/xla/models/segmenter_vit.py': '"""ViT-B/16 with the original Segmenter Mask Transformer decoder."""\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom xla.models.vit import build_vit\nfrom xla.vendor.segmenter.decoder import MaskTransformer\n\n\nclass SegmenterViT(nn.Module):\n    """Binary segmentation using two Segmenter class tokens."""\n\n    def __init__(self, pretrained: bool = True, mask_norm_eps: float = 1e-5):\n        super().__init__()\n        self.backbone = build_vit(pretrained=pretrained, num_classes=2)\n        self.backbone.reset_classifier(0)\n        dim = self.backbone.embed_dim\n        self.decoder = MaskTransformer(\n            n_cls=2, patch_size=16, d_encoder=dim, n_layers=2,\n            n_heads=dim // 64, d_model=dim, d_ff=4 * dim,\n            drop_path_rate=0.0, dropout=0.1,\n        )\n        self.decoder.mask_norm.eps = mask_norm_eps\n\n    def forward(self, images):\n        tokens = self.backbone.forward_features(images)\n        tokens = tokens[:, self.backbone.num_prefix_tokens:, :]\n        logits = self.decoder(tokens, images.shape[-2:])\n        logits = F.interpolate(logits, size=images.shape[-2:],\n                               mode="bilinear", align_corners=False)\n        # Two-class softmax is equivalent to sigmoid(foreground - background).\n        return logits[:, 1:2] - logits[:, 0:1]\n', 'src/xla/vendor/__init__.py': '"""Vendored third-party components."""\n', 'src/xla/vendor/segmenter/__init__.py': '"""Segmenter decoder; see LICENSE and docs/vit/segmenter_source.md."""\n', 'src/xla/vendor/segmenter/blocks.py': '"""\nAdapted from 2020 Ross Wightman\nhttps://github.com/rwightman/pytorch-image-models\n"""\n\nimport torch\nimport torch.nn as nn\nfrom einops import rearrange\nfrom pathlib import Path\n\nimport torch.nn.functional as F\n\nfrom timm.layers import DropPath\n\n\nclass FeedForward(nn.Module):\n    def __init__(self, dim, hidden_dim, dropout, out_dim=None):\n        super().__init__()\n        self.fc1 = nn.Linear(dim, hidden_dim)\n        self.act = nn.GELU()\n        if out_dim is None:\n            out_dim = dim\n        self.fc2 = nn.Linear(hidden_dim, out_dim)\n        self.drop = nn.Dropout(dropout)\n\n    @property\n    def unwrapped(self):\n        return self\n\n    def forward(self, x):\n        x = self.fc1(x)\n        x = self.act(x)\n        x = self.drop(x)\n        x = self.fc2(x)\n        x = self.drop(x)\n        return x\n\n\nclass Attention(nn.Module):\n    def __init__(self, dim, heads, dropout):\n        super().__init__()\n        self.heads = heads\n        head_dim = dim // heads\n        self.scale = head_dim ** -0.5\n        self.attn = None\n\n        self.qkv = nn.Linear(dim, dim * 3)\n        self.attn_drop = nn.Dropout(dropout)\n        self.proj = nn.Linear(dim, dim)\n        self.proj_drop = nn.Dropout(dropout)\n\n    @property\n    def unwrapped(self):\n        return self\n\n    def forward(self, x, mask=None):\n        B, N, C = x.shape\n        qkv = (\n            self.qkv(x)\n            .reshape(B, N, 3, self.heads, C // self.heads)\n            .permute(2, 0, 3, 1, 4)\n        )\n        q, k, v = (\n            qkv[0],\n            qkv[1],\n            qkv[2],\n        )\n\n        attn = (q @ k.transpose(-2, -1)) * self.scale\n        attn = attn.softmax(dim=-1)\n        attn = self.attn_drop(attn)\n\n        x = (attn @ v).transpose(1, 2).reshape(B, N, C)\n        x = self.proj(x)\n        x = self.proj_drop(x)\n\n        return x, attn\n\n\nclass Block(nn.Module):\n    def __init__(self, dim, heads, mlp_dim, dropout, drop_path):\n        super().__init__()\n        self.norm1 = nn.LayerNorm(dim)\n        self.norm2 = nn.LayerNorm(dim)\n        self.attn = Attention(dim, heads, dropout)\n        self.mlp = FeedForward(dim, mlp_dim, dropout)\n        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()\n\n    def forward(self, x, mask=None, return_attention=False):\n        y, attn = self.attn(self.norm1(x), mask)\n        if return_attention:\n            return attn\n        x = x + self.drop_path(y)\n        x = x + self.drop_path(self.mlp(self.norm2(x)))\n        return x\n', 'src/xla/vendor/segmenter/decoder.py': 'import math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom einops import rearrange\n\nfrom timm.layers import trunc_normal_\n\nfrom xla.vendor.segmenter.blocks import Block\nfrom xla.vendor.segmenter.utils import init_weights\n\n\nclass DecoderLinear(nn.Module):\n    def __init__(self, n_cls, patch_size, d_encoder):\n        super().__init__()\n\n        self.d_encoder = d_encoder\n        self.patch_size = patch_size\n        self.n_cls = n_cls\n\n        self.head = nn.Linear(self.d_encoder, n_cls)\n        self.apply(init_weights)\n\n    @torch.jit.ignore\n    def no_weight_decay(self):\n        return set()\n\n    def forward(self, x, im_size):\n        H, W = im_size\n        GS = H // self.patch_size\n        x = self.head(x)\n        x = rearrange(x, "b (h w) c -> b c h w", h=GS)\n\n        return x\n\n\nclass MaskTransformer(nn.Module):\n    def __init__(\n        self,\n        n_cls,\n        patch_size,\n        d_encoder,\n        n_layers,\n        n_heads,\n        d_model,\n        d_ff,\n        drop_path_rate,\n        dropout,\n    ):\n        super().__init__()\n        self.d_encoder = d_encoder\n        self.patch_size = patch_size\n        self.n_layers = n_layers\n        self.n_cls = n_cls\n        self.d_model = d_model\n        self.d_ff = d_ff\n        self.scale = d_model ** -0.5\n\n        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, n_layers)]\n        self.blocks = nn.ModuleList(\n            [Block(d_model, n_heads, d_ff, dropout, dpr[i]) for i in range(n_layers)]\n        )\n\n        self.cls_emb = nn.Parameter(torch.randn(1, n_cls, d_model))\n        self.proj_dec = nn.Linear(d_encoder, d_model)\n\n        self.proj_patch = nn.Parameter(self.scale * torch.randn(d_model, d_model))\n        self.proj_classes = nn.Parameter(self.scale * torch.randn(d_model, d_model))\n\n        self.decoder_norm = nn.LayerNorm(d_model)\n        self.mask_norm = nn.LayerNorm(n_cls)\n\n        self.apply(init_weights)\n        trunc_normal_(self.cls_emb, std=0.02)\n\n    @torch.jit.ignore\n    def no_weight_decay(self):\n        return {"cls_emb"}\n\n    def forward(self, x, im_size):\n        H, W = im_size\n        GS = H // self.patch_size\n\n        x = self.proj_dec(x)\n        cls_emb = self.cls_emb.expand(x.size(0), -1, -1)\n        x = torch.cat((x, cls_emb), 1)\n        for blk in self.blocks:\n            x = blk(x)\n        x = self.decoder_norm(x)\n\n        patches, cls_seg_feat = x[:, : -self.n_cls], x[:, -self.n_cls :]\n        patches = patches @ self.proj_patch\n        cls_seg_feat = cls_seg_feat @ self.proj_classes\n\n        patches = patches / patches.norm(dim=-1, keepdim=True)\n        cls_seg_feat = cls_seg_feat / cls_seg_feat.norm(dim=-1, keepdim=True)\n\n        masks = patches @ cls_seg_feat.transpose(1, 2)\n        masks = self.mask_norm(masks)\n        masks = rearrange(masks, "b (h w) n -> b n h w", h=int(GS))\n\n        return masks\n\n    def get_attention_map(self, x, layer_id):\n        if layer_id >= self.n_layers or layer_id < 0:\n            raise ValueError(\n                f"Provided layer_id: {layer_id} is not valid. 0 <= {layer_id} < {self.n_layers}."\n            )\n        x = self.proj_dec(x)\n        cls_emb = self.cls_emb.expand(x.size(0), -1, -1)\n        x = torch.cat((x, cls_emb), 1)\n        for i, blk in enumerate(self.blocks):\n            if i < layer_id:\n                x = blk(x)\n            else:\n                return blk(x, return_attention=True)\n', 'src/xla/vendor/segmenter/utils.py': '"""Initialization helper from Segmenter; see LICENSE."""\nimport math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom timm.layers import trunc_normal_\n\n\ndef init_weights(m):\n    if isinstance(m, nn.Linear):\n        trunc_normal_(m.weight, std=0.02)\n        if isinstance(m, nn.Linear) and m.bias is not None:\n            nn.init.constant_(m.bias, 0)\n    elif isinstance(m, nn.LayerNorm):\n        nn.init.constant_(m.bias, 0)\n        nn.init.constant_(m.weight, 1.0)\n\ndef resize_pos_embed(posemb, grid_old_shape, grid_new_shape, num_extra_tokens):\n    # Rescale the grid of position embeddings when loading from state_dict. Adapted from\n    # https://github.com/google-research/vision_transformer/blob/00883dd691c63a6830751563748663526e811cee/vit_jax/checkpoint.py#L224\n    posemb_tok, posemb_grid = (\n        posemb[:, :num_extra_tokens],\n        posemb[0, num_extra_tokens:],\n    )\n    if grid_old_shape is None:\n        gs_old_h = int(math.sqrt(len(posemb_grid)))\n        gs_old_w = gs_old_h\n    else:\n        gs_old_h, gs_old_w = grid_old_shape\n\n    gs_h, gs_w = grid_new_shape\n    posemb_grid = posemb_grid.reshape(1, gs_old_h, gs_old_w, -1).permute(0, 3, 1, 2)\n    posemb_grid = F.interpolate(posemb_grid, size=(gs_h, gs_w), mode="bilinear")\n    posemb_grid = posemb_grid.permute(0, 2, 3, 1).reshape(1, gs_h * gs_w, -1)\n    posemb = torch.cat([posemb_tok, posemb_grid], dim=1)\n    return posemb\n\n\n', 'src/xla/vendor/segmenter/LICENSE': 'MIT License\n\nCopyright (c) 2021 Robin Strudel\nCopyright (c) INRIA\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'scripts/continue_segmenter.py': '"""Continue a Segmenter run from its saved epoch-50 weights."""\nimport argparse\nimport hashlib\nimport json\nimport random\nimport shutil\nfrom pathlib import Path\n\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom xla.data.hf_segmentation import make_splits\nfrom xla.engine.vit_segmentation import run_epoch\nfrom xla.models.segmenter_vit import SegmenterViT\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--archive", type=Path, required=True)\n    parser.add_argument("--checkpoint", type=Path, required=True)\n    parser.add_argument("--prior-best", type=Path, required=True)\n    parser.add_argument("--prior-history", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--additional-epochs", type=int, default=50)\n    parser.add_argument("--learning-rate", type=float, default=0.000002)\n    parser.add_argument("--batch-size", type=int, default=4)\n    parser.add_argument("--workers", type=int, default=0)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")\n    args = parser.parse_args()\n    if args.additional_epochs < 1 or args.batch_size < 1 or args.workers < 0:\n        parser.error("additional-epochs and batch-size must be positive; workers nonnegative")\n    if args.learning_rate <= 0:\n        parser.error("learning-rate must be positive")\n    if args.output.exists():\n        raise FileExistsError(f"Output directory already exists: {args.output}")\n    device = torch.device("cuda" if args.device == "auto" and torch.cuda.is_available()\n                          else "cpu" if args.device == "auto" else args.device)\n    if device.type == "cuda" and not torch.cuda.is_available():\n        parser.error("CUDA is not available")\n    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=True)\n    prior_best = torch.load(args.prior_best, map_location="cpu", weights_only=True)\n    history = json.loads(args.prior_history.read_text(encoding="utf-8"))\n    start = checkpoint["epoch"]\n    if start != 50 or [entry["epoch"] for entry in history] != list(range(1, 51)):\n        raise ValueError("Expected the final checkpoint and complete history through epoch 50")\n    if checkpoint.get("settings", {}).get("architecture") != "segmenter_vit_mask_transformer":\n        raise ValueError("Checkpoint is not the previous Mask Transformer run")\n    digest = sha256(args.archive)\n    if checkpoint["archive_sha256"] != digest or prior_best["archive_sha256"] != digest:\n        raise ValueError("Checkpoint was trained on a different archive")\n    previous_winner = max(history, key=lambda record: record["val"]["dice"])\n    if prior_best["epoch"] != previous_winner["epoch"]:\n        raise ValueError("Prior best checkpoint does not match the previous history")\n    if checkpoint["val"] != history[-1]["val"] or prior_best["val"] != previous_winner["val"]:\n        raise ValueError("Checkpoint validation metrics do not match the previous history")\n    if "optimizer" in checkpoint or "optimizer_state_dict" in checkpoint:\n        raise ValueError("Expected the original weights-only checkpoint")\n    settings = dict(checkpoint["settings"])\n    mask_norm_eps = settings["mask_norm_eps"]\n    positive_weight = settings["positive_weight"]\n    random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(args.seed)\n    train_set, val_set, test_set = make_splits(args.archive)\n    print(f"Samples: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}", flush=True)\n    model = SegmenterViT(pretrained=False, mask_norm_eps=mask_norm_eps).to(device)\n    model.load_state_dict(checkpoint["model"], strict=True)\n    del checkpoint, prior_best\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=0.01)\n    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True,\n                              num_workers=args.workers)\n    val_loader = DataLoader(val_set, batch_size=args.batch_size, num_workers=args.workers)\n    settings.update({"start_epoch": start, "epochs": start + args.additional_epochs,\n                     "learning_rate": args.learning_rate, "batch_size": args.batch_size,\n                     "continuation_seed": args.seed, "optimizer_state": "reinitialized",\n                     "source_checkpoint_sha256": sha256(args.checkpoint),\n                     "prior_best_epoch": previous_winner["epoch"]})\n    args.output.mkdir(parents=True, exist_ok=False)\n    shutil.copyfile(args.prior_best, args.output / "best.pt")\n    (args.output / "config.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")\n    (args.output / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")\n    best = previous_winner["val"]["dice"]\n    print(f"Loaded epoch {start}; optimizer initialized anew; previous best "\n          f"epoch={previous_winner[\'epoch\']} dice={best:.6f}", flush=True)\n    for epoch in range(start + 1, start + args.additional_epochs + 1):\n        training = run_epoch(model, train_loader, device, optimizer, positive_weight)\n        validation = run_epoch(model, val_loader, device, positive_weight=positive_weight)\n        record = {"epoch": epoch, "train": training, "val": validation}\n        history.append(record)\n        print(json.dumps(record), flush=True)\n        (args.output / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")\n        output = {"model": model.state_dict(), "epoch": epoch, "val": validation,\n                  "archive_sha256": digest, "settings": settings}\n        if validation["dice"] > best:\n            best = validation["dice"]\n            torch.save(output, args.output / "best.pt")\n        if epoch % 5 == 0 or epoch == start + args.additional_epochs:\n            snapshot = dict(output, optimizer=optimizer.state_dict(),\n                            torch_rng_state=torch.get_rng_state(),\n                            cuda_rng_states=torch.cuda.get_rng_state_all()\n                            if torch.cuda.is_available() else [],\n                            python_rng_state=random.getstate())\n            torch.save(snapshot, args.output / "resume.pt")\n        if epoch == start + args.additional_epochs:\n            torch.save(output, args.output / "final.pt")\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, content in SOURCES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
os.environ['PYTHONPATH'] = str(PROJECT / 'src')
print('Project sources prepared:', len(SOURCES), 'files', flush=True)


In [ ]:
# Resolve only the checked artifacts from the earlier completed Kaggle run.
import hashlib

def checked_input(filename, expected):
    candidates = list(Path('/kaggle/input').rglob(filename))
    for candidate in candidates:
        digest = hashlib.sha256()
        with candidate.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        if digest.hexdigest() == expected:
            return candidate
    raise FileNotFoundError(f'Previous run artifact {filename} with expected SHA-256 not found; candidates={candidates}')

previous_final = checked_input('final.pt', '0fdd1ce119b89b9fd4525ae4d31e7add677ef5c1c4a47e9e5b983d003acc2ac4')
previous_best = checked_input('best.pt', 'f774daa4ececd4266bcf326ba39a89d7f708a6c11d9d649c3f3fbcb7974eb502')
previous_history = checked_input('history.json', '278f59999803b24e8361edaec0b88630c9ad76a3a9ad003c1c29ff6ced66f551')
print('Verified epoch-50 checkpoint, prior best and history.', flush=True)
output = Path('/kaggle/working/segmenter_btxrd_51_100')
command = [sys.executable, '-u', str(PROJECT / 'scripts/continue_segmenter.py'),
           '--archive', str(archive), '--checkpoint', str(previous_final),
           '--prior-best', str(previous_best), '--prior-history', str(previous_history),
           '--output', str(output), '--additional-epochs', '50', '--learning-rate', '0.000002',
           '--batch-size', '4', '--workers', '2', '--seed', '42', '--device', 'cuda']
subprocess.run(command, check=True, env=os.environ.copy())
assert all((output / name).is_file() for name in ('best.pt', 'final.pt', 'resume.pt', 'history.json'))
print('Completed epochs 51 through 100. Output:', [p.name for p in output.iterdir()], flush=True)


best.pt is the best validation checkpoint across epochs 1?100; final.pt is epoch 100. resume.pt also stores optimizer and random states for future recovery. The test split is not used to select checkpoints.
